# 🌳 Phase 2 — Week 5 & 6: Decision Trees + Random Forest
### Project: Social Media Viral Post Predictor

---

**What you'll learn:**
- How Decision Trees work — visually and intuitively
- Why Decision Trees alone can be weak (overfitting)
- Random Forest — how combining many trees fixes the problem
- Feature Importance — which inputs matter most to the model
- Comparing multiple models side by side
- Project: Predict whether a post will go viral

> 💡 Random Forest is one of the most used models in real ML jobs.
> It needs no feature scaling, handles messy data well, and is hard to break.

---
## 🧠 Section 1: How a Decision Tree Works

A Decision Tree makes predictions by asking a series of yes/no questions — exactly like a flowchart.

Here's a manual example for predicting viral posts:

```
Is likes > 5000?
├── YES → Is shares > 200?
│         ├── YES → 🔥 VIRAL
│         └── NO  → ❌ NOT VIRAL
└── NO  → Is hashtag_count > 15?
          ├── YES → 🔥 VIRAL
          └── NO  → ❌ NOT VIRAL
```

The model **automatically finds** the best questions and thresholds from data.
You don't write the rules — it figures them out.

---

### Key advantages over Logistic Regression:
- ✅ No feature scaling needed (likes=50000 and hour=3 are fine as-is)
- ✅ Handles non-linear patterns (not everything is a straight line)
- ✅ Very interpretable — you can literally visualize the tree
- ✅ Works with mixed data types

### The problem with a single Decision Tree:
- ❌ Tends to **overfit** — memorizes training data instead of learning patterns
- A very deep tree will get 100% on training data and fail on new data

**Random Forest solves this.** More on that in Section 4.

---
## 🛠️ Section 2: Build the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 1500

df = pd.DataFrame({
    "likes":           np.random.randint(50, 20000, n),
    "comments":        np.random.randint(5, 2000, n),
    "shares":          np.random.randint(2, 1500, n),
    "hashtag_count":   np.random.randint(0, 30, n),
    "caption_length":  np.random.randint(10, 300, n),
    "hour_posted":     np.random.randint(0, 24, n),
    "follower_count":  np.random.randint(100, 500000, n),
    "is_video":        np.random.randint(0, 2, n),       # 0=image, 1=video
})

# Label: viral = top 20% by engagement
df["engagement"] = df["likes"] + df["comments"] * 2 + df["shares"] * 3
threshold = df["engagement"].quantile(0.80)
df["is_viral"] = (df["engagement"] >= threshold).astype(int)
df = df.drop("engagement", axis=1)

print(f"Dataset shape: {df.shape}")
print(f"Viral posts: {df['is_viral'].sum()} ({df['is_viral'].mean()*100:.1f}%)")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("is_viral", axis=1)
y = df["is_viral"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} rows")
print(f"Test:  {X_test.shape[0]} rows")
# No StandardScaler needed — Decision Trees don't need feature scaling!


---
## 🌱 Section 3: Train a Single Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, accuracy_score, classification_report

# --- Train with NO depth limit (fully grown tree) ---
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

train_f1_full = f1_score(y_train, tree_full.predict(X_train))
test_f1_full  = f1_score(y_test,  tree_full.predict(X_test))

print("=== FULLY GROWN TREE (no depth limit) ===")
print(f"Train F1: {train_f1_full*100:.1f}%")
print(f"Test  F1: {test_f1_full*100:.1f}%")
print(f"Tree depth: {tree_full.get_depth()}")
print()
print("💡 Notice: Train F1 is very high but Test F1 is lower.")
print("   This gap = OVERFITTING. The tree memorized training data.")


In [ ]:
# --- Fix overfitting: limit the tree depth ---
# max_depth controls how many questions the tree can ask

results = []
for depth in [2, 3, 5, 7, 10, 15, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    results.append({
        "max_depth":  str(depth),
        "train_f1":  f1_score(y_train, tree.predict(X_train)),
        "test_f1":   f1_score(y_test,  tree.predict(X_test)),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Plot
plt.figure(figsize=(9, 4))
plt.plot(results_df["max_depth"], results_df["train_f1"], marker="o",
         label="Train F1", color="#6366f1")
plt.plot(results_df["max_depth"], results_df["test_f1"],  marker="o",
         label="Test F1",  color="#ef4444")
plt.title("Overfitting: Train vs Test F1 at Different Depths")
plt.xlabel("max_depth")
plt.ylabel("F1 Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
# 💡 The gap between train and test lines = overfitting gap
# A good depth = where test F1 is highest


In [ ]:
# ✅ Visualize the tree at depth=3 — see exactly what it learned

tree_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_shallow.fit(X_train, y_train)

plt.figure(figsize=(20, 8))
plot_tree(
    tree_shallow,
    feature_names=X.columns.tolist(),
    class_names=["Not Viral", "Viral"],
    filled=True,
    rounded=True,
    fontsize=11
)
plt.title("Decision Tree (depth=3) — Each node shows the question asked", fontsize=14)
plt.show()
# 💡 Read top-down. Each box shows:
# - The question (e.g. shares <= 234)
# - How many samples reach this node
# - The predicted class at this point


---
## 🌲🌲🌲 Section 4: Random Forest — Many Trees, One Answer

A single Decision Tree overfits because it tries to find the single best set of rules.

**Random Forest's idea:**
> Instead of one perfect tree, build 100 different trees — each trained on a
> random subset of data and features. Then let them **vote**.

```
Tree 1:  VIRAL
Tree 2:  NOT VIRAL
Tree 3:  VIRAL
Tree 4:  VIRAL
Tree 5:  VIRAL
──────────────────
Vote:    VIRAL (4 vs 1)  ← final answer
```

Each tree makes different mistakes. But the **majority vote** cancels out individual errors.
Result: much less overfitting, much better generalization.

This idea is called **Ensemble Learning** — combining weak models into a strong one.

Same 3-step sklearn pattern. Just a different class name.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,   # number of trees
    random_state=42,
    n_jobs=-1           # use all CPU cores — trains faster
)
rf_model.fit(X_train, y_train)

rf_train_f1 = f1_score(y_train, rf_model.predict(X_train))
rf_test_f1  = f1_score(y_test,  rf_model.predict(X_test))

print("=== RANDOM FOREST (100 trees) ===")
print(f"Train F1: {rf_train_f1*100:.1f}%")
print(f"Test  F1: {rf_test_f1*100:.1f}%")
print()
print("=== COMPARISON ===")
print(f"Single Tree (full)  — Train: {train_f1_full*100:.1f}%  Test: {test_f1_full*100:.1f}%")
print(f"Single Tree (d=3)   — Train: {f1_score(y_train, tree_shallow.predict(X_train))*100:.1f}%  Test: {f1_score(y_test, tree_shallow.predict(X_test))*100:.1f}%")
print(f"Random Forest       — Train: {rf_train_f1*100:.1f}%  Test: {rf_test_f1*100:.1f}%")


---
## 📊 Section 5: Feature Importance — What Actually Matters?

Random Forest gives you a powerful bonus: it tells you which features
contributed most to the predictions.

This is called **Feature Importance** and it's extremely valuable in real projects —
it tells you what to focus on when collecting data or engineering features.

In [ ]:
# ✅ Feature Importance

importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("=== FEATURE IMPORTANCE (Random Forest) ===")
for feat, score in importances.items():
    bar = "█" * int(score * 100)
    print(f"  {feat:20s}: {score:.4f}  {bar}")

# Plot
plt.figure(figsize=(9, 5))
importances.plot(kind="barh", color="#10b981")
plt.title("Feature Importance — What drives viral posts?")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()


---
## ⚔️ Section 6: Model Comparison — All 3 Side by Side

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score

# Logistic Regression needs scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

# Decision Tree
best_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
best_tree.fit(X_train, y_train)
tree_pred = best_tree.predict(X_test)

# Random Forest (already trained)
rf_pred = rf_model.predict(X_test)

# Compare
models = {
    "Logistic Regression": lr_pred,
    "Decision Tree (d=5)": tree_pred,
    "Random Forest":       rf_pred,
}

print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 70)
for name, pred in models.items():
    print(f"{name:<25} "
          f"{accuracy_score(y_test, pred)*100:>9.1f}% "
          f"{precision_score(y_test, pred)*100:>9.1f}% "
          f"{recall_score(y_test, pred)*100:>9.1f}% "
          f"{f1_score(y_test, pred)*100:>9.1f}%")


In [ ]:
# ✅ Predict on brand new posts

new_posts = pd.DataFrame({
    "likes":          [12000, 300,  5000],
    "comments":       [890,   15,   420],
    "shares":         [650,   8,    180],
    "hashtag_count":  [5,     25,   10],
    "caption_length": [120,   8,    200],
    "hour_posted":    [18,    3,    12],
    "follower_count": [50000, 800,  12000],
    "is_video":       [1,     0,    1],
})

predictions  = rf_model.predict(new_posts)
probabilities = rf_model.predict_proba(new_posts)

post_names = ["Post A (high engagement)", "Post B (low engagement)", "Post C (medium)"]

print("=== VIRAL PREDICTION RESULTS ===")
for name, pred, prob in zip(post_names, predictions, probabilities):
    label = "🔥 VIRAL" if pred == 1 else "❌ NOT VIRAL"
    print(f"{label}  ({prob[1]*100:.1f}% viral probability)")
    print(f"   {name}")
    print()


---
## 🎯 Section 7: Exercises

### Exercise 1
Change `n_estimators` in Random Forest from `100` to `10`, `50`, and `200`.
How does test F1 change? At what point does adding more trees stop helping?

### Exercise 2
Random Forest has a `max_depth` parameter too.
Train two Random Forests: one with `max_depth=5` and one with `max_depth=None`.
Compare train F1 vs test F1 for both. Is Random Forest with unlimited depth
still better than a single unlimited-depth tree? Why?

### Exercise 3
Add two new engineered features to the dataset before training:
```python
df["engagement_rate"] = (df["likes"] + df["comments"]) / (df["follower_count"] + 1)
df["shares_per_like"] = df["shares"] / (df["likes"] + 1)
```
Retrain Random Forest and check the new feature importance chart.
Did these engineered features land in the top 3?

> 💡 `engagement_rate` is a metric real social media platforms use internally.
> You're building industry-standard features here.

In [ ]:
# 🏋️ YOUR WORKSPACE

# ✏️ Exercise 1:


# ✏️ Exercise 2:


# ✏️ Exercise 3:



---
## ✅ Week 5–6 Checklist

- [ ] Can explain how a Decision Tree makes decisions (flowchart analogy)
- [ ] Understand what overfitting is and how `max_depth` controls it
- [ ] Can explain Random Forest as majority vote of many trees
- [ ] Know why Random Forest generalizes better than a single tree
- [ ] Can read and interpret Feature Importance chart
- [ ] Compared 3 models side by side on same dataset
- [ ] Completed all 3 exercises

---

## 🎉 Phase 2 Core Complete!

You now know the 3 foundational ML models:

| Model | Use when |
|---|---|
| Linear Regression | Predict a number |
| Logistic Regression | Predict yes/no, needs scaling |
| Random Forest | Most real problems — robust, no scaling, powerful |

---

## 🔜 What's Next — Phase 2 Final: Model Evaluation + Scikit-learn Pipelines

Before moving to Phase 3 (Deep Learning), you need one more week:
- **Cross-validation** — a smarter way to evaluate than a single train/test split
- **Hyperparameter tuning** — automatically finding the best model settings
- **Sklearn Pipeline** — wrapping preprocessing + model into one clean object
- This is how **production ML code** actually looks

> 💬 Finish the exercises, come back with your solutions and let's wrap up Phase 2!